# Classification I: the predictive workflow

*The train/test workflow, logistic regression, KNN, and error costs*

With this chapter we begin Part IV, the third stage of the course map: predicting what is likely. Prairie Wholesale loses roughly one out of five customers every year. If the account team knew in advance which customers were at risk, it could call them before they left. This is a **classification** problem, because a category, in this case churn or stay, is predicted for each customer. In this chapter we build our first prediction models, which we use to identify the customers who are likely to churn. We also learn to measure a model's accuracy on customers it has not seen. Any claim about how well a model predicts is measured in this way.

You can run this chapter's notebook in two ways. In the cloud, [**open ch-09-classification-1.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-09-classification-1.ipynb); nothing needs to be installed. Locally, use the `pyba-core` environment (Appendix A). Either way, run the setup cell below first. No keys or paid accounts are needed anywhere in this book; Colab requires only a free Google account.


In [ ]:
# Setup. Run this cell once per session. It installs this chapter's
# packages; on Google Colab it also fetches the course data.
%pip install -q pandas plotly scikit-learn
import sys
if "google.colab" in sys.modules:
    !test -d pyba-companion || git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

| Tool | Why we use it here | Alternatives | Trade-off |
|---|---|---|---|
| scikit-learn | The standard machine-learning library, with one fit/predict interface across dozens of models | statsmodels (inference tables, fewer models), XGBoost (one model family) | scikit-learn is built around prediction, the focus of Part IV; statsmodels remains the choice for coefficient inference |

## The prediction problem
In Chapters [5](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-05-relationships.html), [7](https://pyba.murtaza.cc/parts/part-03-statistical/ch-07-inference.html) and [8](https://pyba.murtaza.cc/parts/part-03-statistical/ch-08-regression.html) we worked with the **customer snapshot**, a table with one row per customer that summarizes each customer's ordering history up to a fixed date (order counts, revenue, weeks since the last order, tenure, etc.). In [Chapter 8](https://pyba.murtaza.cc/parts/part-03-statistical/ch-08-regression.html) we used it to model revenue. In this chapter we use the same table to predict a different column, `churned`, which records whether the customer stopped ordering from Prairie Wholesale.

In [ ]:
import pandas as pd
import plotly.express as px

from pyba import DATA_DIR

snap = pd.read_csv(DATA_DIR / "pw_customer_snapshot.csv")
snap.head(3)

Each column of the snapshot was computed from one of three time periods around a cutoff date, December 28, 2025, as shown in the table below. In prediction, the columns used to make a prediction are called **features**, or input variables (in Chapter 8 we called them predictors), and the column to be predicted is called the **label**. The label, `churned`, is 1 if the customer placed no order in the 26 weeks after the cutoff.

| Period | Dates | Snapshot columns |
|---|---|---|
| Before the cutoff | up to December 28, 2025 | `orders_12m`, `revenue_12m`, `weeks_since_last_order`, `tenure_months`, etc. |
| Cutoff | December 28, 2025 | none; the customers are scored on this day |
| After the cutoff | December 29, 2025 to June 28, 2026 | `churned` |

Because the features are measured before the cutoff and the label after it, a model trained on this table is given only what was known at the time of prediction, as it will be in use. In the first row, customer C0001 ordered 0.9 weeks before the cutoff and did not churn. Customer C0002 had not ordered for 21.9 weeks and did churn.

In [ ]:
snap["churned"].value_counts(normalize=True).round(3)

Of the active customers, 22% churned. From this number we can build our first model: predict "stay" for every customer. This model is correct 78% of the time. It is also useless, because it never flags a single at-risk customer. A model that predicts the most common outcome for every customer is called a **naive model**, and its accuracy, here 78%, is the **majority baseline**. A useful model must exceed this baseline.

From the naive model, we can also see that accuracy alone is a poor measure of a model's performance. A model that never identifies a churner is still almost 80% accurate, because churners are only about one customer in five. When one outcome is much rarer than the other, the data are said to have **class imbalance**. With class imbalance, a high accuracy can be reached by predicting the common outcome every time. We return to this point with the confusion matrix.

The naive model uses none of the customers' features. Before we fit a model that does, we look at one feature on its own: recency, the number of weeks since a customer's last order.

In [ ]:
px.histogram(snap, x="weeks_since_last_order", color=snap["churned"].astype(str),
             barmode="overlay", nbins=40, opacity=0.6,
             labels={"weeks_since_last_order": "Weeks since last order (at the cutoff)",
                     "color": "churned"})

In the chart code, the 0/1 label is converted to the text values "0" and "1" with `.astype(str)`, so that the two groups are drawn by Plotly as categories, each with its own color.

The median customer who stayed had last ordered 2.4 weeks before the cutoff. The median churner had last ordered 13.5 weeks before it. Recency is therefore informative. Because the two histograms overlap, however, a rule based on recency alone would misclassify many customers. To build a more useful model, we will combine recency with the other features.

### Classification as drawing a boundary
With two features (input variables), we can represent each customer as a point on a chart, with the color describing the outcome for that customer. In this way, we can see the task of a classification model directly. The chart below shows 29 made-up customers, placed by weeks since the last order and orders in the last 12 months. Most churners (red) are at the lower right: they have not ordered for a while, or they order rarely. A classification model divides the chart into a churn region and a stay region. The line between the two regions is the **decision boundary**.

The online edition has an interactive explorer here. [Open the interactive explorer](https://pyba.murtaza.cc/assets/demos/decision-boundary.html).




In [ ]:
#@title Run this cell to draw the figure (plotting code only)
import numpy as np
import plotly.graph_objects as go

# Hand-placed customers: weeks since last order, orders in the last 12 months, churned (1) or stayed (0)
toy = pd.DataFrame(
    [(1, 20, 0), (2, 14, 0), (3, 22, 0), (2, 8, 0), (4, 17, 0), (5, 25, 0), (6, 19, 0), (3, 4, 0),
     (7, 24, 0), (8, 22, 0), (5, 11, 0), (9, 27, 0), (1, 3, 0), (8, 9, 0), (10, 15, 0),
     (7, 2, 1), (8, 5, 1), (10, 8, 1), (12, 8, 1), (12, 17, 1), (14, 18, 1), (15, 12, 1),
     (13, 4, 1), (16, 22, 1), (18, 16, 1), (17, 7, 1), (9, 2, 1),
     (6, 8, 1), (11, 12, 0)],
    columns=["weeks", "orders", "churned"])

line_x = np.array([4.8, 16.8])                       # where the line enters and leaves the chart
fig = go.Figure()
fig.add_trace(go.Scatter(x=[4.8, 16.8, 20, 20, 4.8], y=[0, 30, 30, 0, 0], fill="toself",
                         fillcolor="#f6d5cc", line_width=0, hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=line_x, y=2.5 * line_x - 12, mode="lines", line=dict(color="#333", width=2),
                         name="orders = 2.5 × weeks − 12"))
for label, color, name in [(0, "#2b6cb0", "stayed"), (1, "#c0392b", "churned")]:
    pts = toy[toy["churned"] == label]
    fig.add_trace(go.Scatter(x=pts["weeks"], y=pts["orders"], mode="markers", name=name,
                             marker=dict(size=9, color=color, line=dict(width=1, color="white"))))
# Two new customers, not in the data: each is classified by the side of the line it falls on
new = pd.DataFrame({"weeks": [14, 7], "orders": [10, 14], "name": ["A", "B"]})
new["predicted"] = (new["orders"] < 2.5 * new["weeks"] - 12).map({True: "churn", False: "stay"})
fig.add_trace(go.Scatter(x=new["weeks"], y=new["orders"], mode="markers+text", name="new customer",
                         text=[f"new customer {n}: predicted {p}" for n, p in zip(new["name"], new["predicted"])],
                         textposition=["bottom right", "top right"],
                         marker=dict(size=14, color="white", line=dict(width=3, color="#333"))))
fig.update_xaxes(title_text="Weeks since last order", range=[0, 20])
fig.update_yaxes(title_text="Orders, last 12 months", range=[0, 30])
fig.update_layout(height=380, plot_bgcolor="#dbe9f6", legend=dict(orientation="h", y=-0.25))
fig


The simplest boundary is a straight line. The model in the chart predicts churn for every customer below the line

$$\text{orders} = 2.5 \times \text{weeks} - 12$$

For example, a customer who last ordered 10 weeks ago is predicted to churn with fewer than 2.5 × 10 − 12 = 13 orders. This one rule classifies 27 of the 29 customers correctly. The two exceptions, a churner who ordered six weeks before and a customer who stayed despite an 11-week gap, lie on the wrong side of the line.

Each model in this chapter has a decision boundary of this kind. Fitting a model means choosing its boundary from the training data, and models differ in how flexible a boundary they can draw. With all 20 features (input variables) of the training data, we can no longer visualize the boundary as a line on a chart. A straight boundary then becomes the high-dimensional equivalent of a line, a **hyperplane**, but its role is the same: to split the input data into the outcome classes, or in other words, to classify the data. We can then use the decision boundary to classify new, unseen customers. A new customer who falls in the red region is predicted to churn, and one who falls in the blue region is predicted to stay.

The online edition has an interactive explorer here.


## The train/test discipline

**scikit-learn** is the standard machine-learning library of the Python ecosystem. It is an open-source project built around one interface that every model shares: we create a model object, `fit` it on training data, and `predict` on new data. Because scikit-learn models require numeric inputs, we first convert the text columns:

In [ ]:
# Features: every column except the ID, the pilot flag, and the label itself.
# get_dummies replaces each text column with one True/False column per category.
X = pd.get_dummies(snap.drop(columns=["customer_id", "pilot", "churned"]),
                   columns=["business_type", "region"])
y = snap["churned"]

X.shape

`X` is the **feature table**, the columns the model uses to make its predictions, and `y` is the column of labels, or **targets**, that we want the model to learn to predict. Three columns are left out of `X`. `customer_id` is only an identifier, and `churned` is the label itself. `pilot` is dropped because the pilot ran from March to May 2026, after the cutoff, so a customer's pilot assignment was not known at the time of prediction. This follows a rule that applies to every predictive model: a feature may use only information that is available on or before the day the prediction is made. With a feature that breaks this rule, the model looks far better in testing than it will be in use, as we show in Chapter 9.

`get_dummies` converts the two text columns, `business_type` and `region`, into dummy variables, with one True/False column per category. These are the dummy variables of [Chapter 8](https://pyba.murtaza.cc/parts/part-03-statistical/ch-08-regression.html), where they were created automatically by statsmodels whenever a formula contained `C()`. Here they appear as visible columns of `X`. The 8 numeric columns are kept as they are, and the 2 text columns are replaced by 12 dummy columns (six business types and six regions), for 20 columns in total.

In [ ]:
# --- Look: two of the new indicator columns for the first three customers ---
X[["business_type_nonprofit", "business_type_retail"]].head(3)

The first two customers are retail, so `business_type_retail` is True in their rows. The third customer is a nonprofit. In scikit-learn, True is treated as 1 and False as 0.

Before fitting anything, we split the customers into two sets: a **training set** and a **test set**. The model learns from the training set, while the test set is held back and not used in the learning process. The split is made with the function `train_test_split`. In the first line of the code below, it is imported from scikit-learn's `model_selection` module. The library groups its tools into modules by purpose. In this chapter, each import names one of these modules:

- `model_selection`: splitting the data
- `linear_model` and `neighbors`: the two kinds of model
- `preprocessing`: scaling
- `pipeline`: chaining steps
- `metrics`: scoring

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=0)

print(X_train.shape, X_test.shape)
print(f"churn rate: training {y_train.mean():.3f}, test {y_test.mean():.3f}")

`train_test_split` returns four objects. They come in a fixed order: the training features, the test features, the training labels, and the test labels. We need to pay attention to three of its arguments:

- **`test_size=0.3`.** With this argument, 30% of the customers are held out: 262 for testing and 610 for training.
- **`stratify=y`.** With this argument, the split is **stratified**: the churners and the customers who stayed are each split 70/30 separately, so the churn rate is the same in both sets (22.0% and 22.1% above). This is important with class imbalance. Without it, a small test set could receive too few churners by chance, and the test results would then be a poor guide to how the model handles churners.
- **`random_state=0`.** With this argument, the split is the same on every run. Your numbers will therefore match the numbers in this book.

We give up 30% of the data for an important reason: to test whether the model has learned patterns that carry over to new customers or has only memorized the answers for the customers it was trained on. A model scored on its own training data has already seen the answers. On the test set, we measure the model's performance on customers absent from its training data, which is the situation of every customer the model will score in practice. In Part III we asked whether an association was distinguishable from chance. In Part IV we ask whether a prediction is accurate on data the model has not seen.

### Underfitting and overfitting
A model can fail on new data in two opposite ways. A model that is too simple **underfits**: it misses real patterns in the data, so its accuracy is low on both the training set and the test set. A model that is too flexible **overfits**: it learns the noise in the training data along with the patterns, so its accuracy is high on the training set and much lower on the test set. A useful model lies between the two.

The two failures are also described in terms of **bias** and **variance**. An underfitted model has high bias: its errors are systematic, and they remain the same whichever training customers it is given. An overfitted model has high variance: its predictions change a great deal when it is fitted on a different sample of customers. Making a model more flexible lowers its bias and raises its variance. Our goal is to balance the two, so that the model learns the useful patterns in the data. This is the **bias-variance trade-off**. It also means that our goal is not to predict every training customer correctly; a model that does so has overfitted. If we blindly maximize the training accuracy, the model learns the noise in the data, that is, the variation that the features cannot explain. In real data, the classes are rarely separated cleanly, and some noise is always present.

The online edition has an interactive explorer here. [Open the interactive explorer](https://pyba.murtaza.cc/assets/demos/fit-variance.html).


The figure shows the three cases for the customers of the figure.

In [ ]:
#@title Run this cell to draw the figure (plotting code only)
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Hand-placed customers: weeks since last order, orders in the last 12 months, churned (1) or stayed (0).
# The last two are exceptions: a churner at (6, 8) and a customer who stayed at (11, 12).
points = pd.DataFrame(
    [(1, 20, 0), (2, 14, 0), (3, 22, 0), (2, 8, 0), (4, 17, 0), (5, 25, 0), (6, 19, 0), (3, 4, 0),
     (7, 24, 0), (8, 22, 0), (5, 11, 0), (9, 27, 0), (1, 3, 0), (8, 9, 0), (10, 15, 0),
     (7, 2, 1), (8, 5, 1), (10, 8, 1), (12, 8, 1), (12, 17, 1), (14, 18, 1), (15, 12, 1),
     (13, 4, 1), (16, 22, 1), (18, 16, 1), (17, 7, 1), (9, 2, 1),
     (6, 8, 1), (11, 12, 0)],
    columns=["weeks", "orders", "churned"])

xs = np.linspace(0, 20, 401)

def curve(x):                       # the pattern: churn below this curve
    return 0.25 * np.clip(x - 3, 0, None) ** 2

# The too-flexible boundary: the curve with zigzags, two of which reach out to the exceptions
zx = [0, 3, 4, 5, 5.5, 6, 6.5, 7, 7.5, 8, 9, 9.5, 10, 10.5, 11, 11.5, 12, 13, 14, 20]
zy = [0, 0, 1.5, 0.5, 2, 9.5, 2.5, 6, 4, 7.5, 7, 11, 10, 15, 10.5, 16, 19, 18, 26, 30 + 60]
def zigzag(x):
    return np.interp(x, zx, zy)

panels = [("Underfitting: too simple", None), ("A good fit", curve), ("Overfitting: too flexible", zigzag)]
fig = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.03,
                    subplot_titles=[t for t, _ in panels])
for col, (title, boundary) in enumerate(panels, start=1):
    if boundary is None:            # too simple: one straight cut on recency alone, at 12.5 weeks
        bx, by = [12.5, 12.5], [0, 30]
        predicted = (points["weeks"] > 12.5).astype(int)
        fig.add_shape(type="rect", x0=12.5, x1=20, y0=0, y1=30, fillcolor="#f6d5cc", line_width=0, layer="below", row=1, col=col)
    else:
        by = np.minimum(boundary(xs), 30)
        bx = xs
        predicted = (points["orders"] < boundary(points["weeks"])).astype(int)
        fig.add_trace(go.Scatter(x=np.r_[xs, xs[::-1]], y=np.r_[by, np.zeros_like(xs)], fill="toself",
                                 fillcolor="#f6d5cc", line_width=0, hoverinfo="skip", showlegend=False), row=1, col=col)
    fig.add_trace(go.Scatter(x=bx, y=by, mode="lines", line=dict(color="#333", width=2),
                             hoverinfo="skip", name="boundary", legendgroup="b", showlegend=(col == 1)), row=1, col=col)
    for label, color, name in [(0, "#2b6cb0", "stayed"), (1, "#c0392b", "churned")]:
        pts = points[points["churned"] == label]
        fig.add_trace(go.Scatter(x=pts["weeks"], y=pts["orders"], mode="markers",
                                 marker=dict(size=9, color=color, line=dict(width=1, color="white")),
                                 name=name, legendgroup=name, showlegend=(col == 1)), row=1, col=col)
    errors = int((predicted != points["churned"]).sum())
    fig.layout.annotations[col - 1].text = f"{title}<br><sup>{errors} of {len(points)} training customers misclassified</sup>"
fig.update_xaxes(title_text="Weeks since last order", range=[0, 20])
fig.update_yaxes(range=[0, 30])
fig.update_yaxes(title_text="Orders, last 12 months", row=1, col=1)
fig.update_layout(height=400, margin=dict(t=80), plot_bgcolor="#dbe9f6", legend=dict(orientation="h", y=-0.22))
fig

## Logistic regression

With the data split and the goal of a balanced fit in mind, we can now construct our first model on the real customers. We start with a **logistic regression** model, which is an adaptation of the [Chapter 8](https://pyba.murtaza.cc/parts/part-03-statistical/ch-08-regression.html) linear model to yes/no outcomes. The model predicts in two steps:

1. As in linear regression, each customer receives a **score**: an intercept plus each feature times its coefficient.
2. The score is then converted into a probability between 0 and 1 with the **logistic function**, $p = 1 / (1 + e^{-\text{score}})$.

The score on its own is a straight line, as in linear regression. It can take any value, from large negative numbers to large positive ones, and many of these values make no sense as probabilities. For example, a linear model alone could predict a churn probability of 1.4 or of −0.2. We therefore apply the logistic function to the score. This logistic transformation converts every score into a value between 0 and 1, which can be read as a probability. The figure shows the two steps side by side for a model with a single feature, weeks since the last order.

In [ ]:
#@title Run this cell to draw the figure (plotting code only)
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

b0, b1 = -3.06, 0.22                       # the one-feature model in the explorer below
weeks = np.linspace(0, 48, 200)
score = b0 + b1 * weeks
marked = np.array([5, 14, 30])            # three customers, marked in both panels

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=["Step 1: the score", "Step 2: the probability"])
fig.add_trace(go.Scatter(x=[0, 48, 48, 0, 0], y=[0, 0, 1, 1, 0], fill="toself", fillcolor="rgba(130,130,130,0.35)", mode="lines",
                         line_width=0, hoverinfo="skip", showlegend=False), row=1, col=1)   # the range of a probability
fig.add_annotation(x=47, y=1, text="0 to 1", showarrow=False, xanchor="right",
                   yanchor="bottom", font=dict(size=11), row=1, col=1)
fig.add_trace(go.Scatter(x=weeks, y=score, mode="lines", line=dict(color="#8250df", width=2.5),
                         showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=weeks, y=1 / (1 + np.exp(-score)), mode="lines", line=dict(color="#c0392b", width=2.5),
                         showlegend=False), row=1, col=2)
fig.add_hline(y=0.5, line_dash="dash", line_color="gray", row=1, col=2)
for col, values in [(1, b0 + b1 * marked), (2, 1 / (1 + np.exp(-(b0 + b1 * marked))))]:
    fig.add_trace(go.Scatter(x=marked, y=values, mode="markers+text", showlegend=False,
                             text=[f"{v:.2f}" for v in values], textposition="middle right" if col == 1 else "bottom right",
                             marker=dict(size=9, color="#333")), row=1, col=col)
fig.update_xaxes(title_text="Weeks since last order", range=[0, 48])
fig.update_yaxes(title_text="score", range=[-4, 8], row=1, col=1)
fig.update_yaxes(title_text="probability of churn", range=[-0.05, 1.05], row=1, col=2)
fig.update_layout(height=340, margin=dict(t=40, l=50, r=20))
fig

A large positive score becomes a probability near 1, a large negative score becomes a probability near 0, and a score of exactly 0 becomes a probability of 0.5. The customers with a score of 0 therefore form the model's decision boundary, which is a straight line, as in Chapter 9 (with 20 features, a hyperplane).

The online edition has an interactive explorer here: a one-feature logistic model of churn on recency, with the linear score and the resulting probability curve drawn side by side, and sliders for the intercept and slope. [Open the interactive explorer](https://pyba.murtaza.cc/assets/demos/logistic-curve.html).


We put the model in a **pipeline** with a `StandardScaler` in front of it. A pipeline is a chain of steps that we use to build a model. `make_pipeline` returns a pipeline made of the steps we list, in that order, and `LogisticRegression` is scikit-learn's logistic regression model. When we call `fit` or `predict` on a pipeline, its steps are run in order. With a **`StandardScaler`**, every feature is rescaled to mean 0 and standard deviation 1: the feature's training-set mean is subtracted, and the result is divided by its training-set standard deviation. After scaling, revenue (in dollars) and recency (in weeks) are measured on the same scale. The cell below shows the scaling for two features of the first three training customers:

In [ ]:
from sklearn.preprocessing import StandardScaler

# --- Look: two features of the first three training customers, before and after scaling ---
pair = X_train[["revenue_12m", "weeks_since_last_order"]]
scaler = StandardScaler().fit(pair)             # computes each feature's training mean and standard deviation
scaled = scaler.transform(pair)                 # (value - mean) / standard deviation, for every customer
print(pd.DataFrame({"mean": scaler.mean_, "standard deviation": scaler.scale_},
                   index=pair.columns).round(2))
pd.DataFrame({"revenue_12m": pair["revenue_12m"].head(3), "revenue, scaled": scaled[:3, 0],
              "weeks_since_last_order": pair["weeks_since_last_order"].head(3),
              "weeks, scaled": scaled[:3, 1]}).round(2)

The second customer last ordered 19.4 weeks before the cutoff. The scaled value is (19.4 − 6.36) / 6.95 = 1.88: this customer's recency is 1.88 standard deviations above the training mean. The third customer's revenue, \$15,948, becomes (15,948 − 7,303) / 9,212 = 0.94. After scaling, a value of 1 means one standard deviation above the mean, whatever the original units. Scaling has some effect on how the logistic model is fitted and a much larger effect on KNN, later in this chapter.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
logit.fit(X_train, y_train)

train_accuracy = accuracy_score(y_train, logit.predict(X_train))
test_accuracy = accuracy_score(y_test, logit.predict(X_test))
print(f"training accuracy {train_accuracy:.3f}")
print(f"test accuracy     {test_accuracy:.3f}  (majority baseline 0.780)")

When we call `logit.fit`, the scaler is fitted on the training features first, and the logistic model is then fitted on the scaled features. When we call `logit.predict`, we give it the values of the input variables for a set of customers, such as the test customers in `X_test`. The same scaling, with the training means and standard deviations, is applied to these values, and we get a 0 (stay) or 1 (churn) for each of these customers.

Behind this 0 or 1 are two steps. The model first computes each customer's probability of churn, a number between 0 and 1. This probability is then converted into a category with a **threshold**: a customer whose probability is above the threshold is predicted to churn (1), and every other customer is predicted to stay (0). In scikit-learn, as in most software, the default threshold is 0.5. A probability of 0.5 corresponds to a score of 0, so with this threshold the predictions follow the decision boundary described above. We return to the choice of threshold in Chapter 9.

`accuracy_score` returns the share of predictions that match the actual labels. With `max_iter=2000`, more fitting steps are allowed than the default, so that no warning is raised.

The model is correct for 87% of the training customers and for 84% of the test customers. The training number is higher because the model was fitted to those customers. The test accuracy is the number we report, because it is a measure of the model's performance on customers it has not seen. The model's test accuracy is about 6 percentage points above the 78% baseline. To judge the business value of those 6 points, we need the confusion matrix, which we construct later in this chapter.

From the fitted coefficients, we can see how each feature is associated with the score:

In [ ]:
# logit[-1] is the last step of the pipeline: the fitted LogisticRegression.
# Its coef_ holds one coefficient per (scaled) feature.
coefs = pd.Series(logit[-1].coef_[0], index=X.columns).sort_values()
coefs.round(2)

A pipeline can be indexed like a Python list, so `logit[-1]` is its last step, the fitted logistic model. Its `coef_` attribute holds the coefficients in a two-dimensional array with one row, and with `[0]` we select that row. Because the features were scaled, each coefficient is the change in the score for one standard deviation more of that feature. `weeks_since_last_order` has by far the largest coefficient (1.64). A customer who has not ordered for longer receives a much higher churn score. Higher values of `revenue_12m` and `avg_order_value` are associated with a lower score. As in Chapter 8, these coefficients describe association. They do not show that any feature causes churn.

## K-nearest neighbors
**K-nearest neighbors (KNN)** predicts by analogy. To classify a customer, we find the k most similar customers in the training data and take a vote among them. With k = 15, if 9 of a customer's 15 nearest training customers churned, the customer is predicted to churn. KNN has no equation and no coefficients. The training data itself is the model, and the decision boundary has a bend wherever the vote of nearby customers changes, as in the too-flexible panel of Chapter 9. A model of this type is called a **lazy learner**: it has no explicit training stage other than storing the data. At the time of prediction, the nearest neighbors are found with a **distance metric**, a formula for the distance between two customers. The most common metric, and the default in scikit-learn, is the **Euclidean distance**: the ordinary straight-line distance between two points, computed across all features. Other metrics, such as the Manhattan distance, are also available.

We include KNN because the role of distance is explicit in it. In KNN, "similar" means close in the feature space. Because distances are computed from the raw values, they depend on the units of each feature. Without scaling, a revenue difference of \$1,000 would count a thousand times as much as a recency difference of 1 week. Revenue would then account for almost all of every distance.

Because KNN is also available in scikit-learn, training a KNN model is as simple as changing the model and its parameters in our pipeline: we replace `LogisticRegression` with `KNeighborsClassifier`, and we set the number of neighbors, k, with its parameter `n_neighbors`. To see the effect of scaling, we also fit the same model without the scaler:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))
knn.fit(X_train, y_train)
print(f"KNN, scaled:   test accuracy {accuracy_score(y_test, knn.predict(X_test)):.3f}")

knn_raw = KNeighborsClassifier(n_neighbors=15)    # the same model with no scaler
knn_raw.fit(X_train, y_train)
print(f"KNN, unscaled: test accuracy {accuracy_score(y_test, knn_raw.predict(X_test)):.3f}")

With scaling, KNN reaches 83.2%, about one percentage point below the logistic model. Without scaling, it reaches 77.1%, which is below the 78% majority baseline. In the unscaled model, customers are compared almost entirely by revenue and average order value, the two features measured in large numbers of dollars.

The online edition has an interactive explorer here: the training customers plotted on recency and revenue, with a clickable new customer, its k nearest neighbors, and a switch for feature scaling. [Open the interactive explorer](https://pyba.murtaza.cc/assets/demos/knn-neighbors.html).


The setting k can lie anywhere between two extremes. We compute the training and test accuracy at both extremes and at two values between them:

In [ ]:
rows = []
for k in [1, 15, 151, 610]:
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    model.fit(X_train, y_train)
    rows.append([k, accuracy_score(y_train, model.predict(X_train)),
                 accuracy_score(y_test, model.predict(X_test))])

pd.DataFrame(rows, columns=["k", "training accuracy", "test accuracy"]).round(3)

At k = 1, each training customer's nearest neighbor is that customer itself. The training accuracy is therefore 100%. The test accuracy, 78.6%, is no better than the baseline: the model has memorized the training data, noise included. At k = 610, the size of the whole training set, every prediction is the vote of all 610 training customers. Every customer is then predicted to stay, as in the majority baseline. Between the extremes, the accuracy at k = 15 is similar on both sets (84.4% and 83.2%). These are the cases of Chapter 9, now on the real data. A large gap between training and test accuracy, as at k = 1, is a sign of overfitting, and low accuracy on both sets, as at k = 610, is a sign of underfitting. Notice that, while trying to determine the best number of neighbors, that is, the value of k, we looked at the test accuracy and so used the test set. In practice, you should never do this. If the test set is used to make a decision about the model, it becomes, in a sense, part of the training process: the chosen k is tuned to the test customers, and the test accuracy can no longer be used as an indicator of the model's performance on unseen data. In [Chapter 10](https://pyba.murtaza.cc/parts/part-04-predictive/ch-10-classification-2.html) we learn a systematic way to choose settings such as k without using the test set.

## The confusion matrix

With accuracy, we count only how often the model is right. A prediction, however, can be right or wrong in two ways each. Depending on whether a customer actually churned and whether the model predicted churn, there are four possible outcomes: a churner who is flagged, a churner who is missed, a customer who stayed but is flagged, and a customer who stayed and is left alone. In the accuracy number, these four outcomes are reduced to one figure, the share of customers in the first and last groups. As we saw with the naive model, this figure can be high under class imbalance even when no churner is flagged. A **confusion matrix** is a table in which the four outcomes are counted separately:

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, logit.predict(X_test))
pd.DataFrame(cm, index=["actually stayed", "actually churned"],
             columns=["predicted stay", "predicted churn"])

`confusion_matrix` returns a small NumPy array of counts, with the actual labels in the rows and the predictions in the columns. To label the rows and columns, we wrap it in a DataFrame. Each cell has a standard name:

| Cell | Name | Count | Meaning for Prairie Wholesale |
|---|---|---|---|
| actually churned, predicted churn | **true positive** (TP) | 25 | a churner the model flagged |
| actually churned, predicted stay | **false negative** (FN) | 33 | a churner the model missed |
| actually stayed, predicted churn | **false positive** (FP) | 8 | a false alarm: an offer sent to a customer who would have stayed |
| actually stayed, predicted stay | **true negative** (TN) | 196 | a loyal customer correctly left alone |

"Positive" refers to the outcome we are trying to detect, churn. We read the bottom row first, since it holds the churners. Of the 58 test customers who churned, the model flagged 25 and missed 33. The share of actual churners the model flagged, 25 of 58, is the **recall** (43%). Of the 33 customers the model flagged, 25 actually churned. This share is the **precision** (76%).

Two more names are common. **Sensitivity** is another name for recall. **Specificity** is the same share for the customers who stayed: of the 204 customers who stayed, the model correctly left 196 alone, a specificity of 96%. The figure shows where each of these metrics comes from in the matrix.

In [ ]:
#@title Run this cell to draw the figure (plotting code only)
import plotly.graph_objects as go

# (row, column) positions: rows 2 = actually stayed, 1 = actually churned, 0 = total;
# columns 0 = predicted stay, 1 = predicted churn, 2 = total
cells = {(2, 0): ("TN", 196), (2, 1): ("FP", 8), (2, 2): ("", 204),
         (1, 0): ("FN", 33), (1, 1): ("TP", 25), (1, 2): ("", 58),
         (0, 0): ("", 229), (0, 1): ("", 33), (0, 2): ("", 262)}
fills = {2: "#dbe9f6", 1: "#f6d5cc", 0: "#eef0f2"}          # blue: stayed, red: churned, gray: totals
fig = go.Figure()
for (r, c), (name, n) in cells.items():
    fill = "#eef0f2" if c == 2 else fills[r]
    fig.add_shape(type="rect", x0=c + 0.04, x1=c + 0.96, y0=r + 0.04, y1=r + 0.96,
                  fillcolor=fill, line_width=0, layer="below")
    label = f"{name}<br><b>{n}</b>" if name else f"<b>{n}</b>"
    fig.add_annotation(x=c + 0.5, y=r + 0.5, text=label, showarrow=False, font=dict(size=20))
for c, label in enumerate(["predicted stay", "predicted churn", "total"]):
    fig.add_annotation(x=c + 0.5, y=3.06, text=label, showarrow=False, yanchor="bottom")
for r, label in [(2, "actually stayed"), (1, "actually churned"), (0, "total")]:
    fig.add_annotation(x=-0.04, y=r + 0.5, text=label, showarrow=False, xanchor="right")
# specificity and recall: one row each; precision: one column
for r, color, label in [(2, "#2b6cb0", "<b>Specificity</b><br>TN / (TN + FP)<br>= 196 / 204 = 96%"),
                        (1, "#c0392b", "<b>Recall (sensitivity)</b><br>TP / (TP + FN)<br>= 25 / 58 = 43%")]:
    fig.add_shape(type="rect", x0=0.0, x1=3.0, y0=r + 0.02, y1=r + 0.98, line=dict(color=color, width=3))
    fig.add_annotation(x=3.1, y=r + 0.5, text=label, showarrow=False, xanchor="left", align="left",
                       font=dict(size=13, color=color))
fig.add_shape(type="rect", x0=1.09, x1=1.91, y0=-0.08, y1=2.99, line=dict(color="#333", width=2, dash="dash"))
fig.add_annotation(x=1.5, y=-0.12, text="<b>Precision</b> = TP / (TP + FP) = 25 / 33 = 76%", showarrow=False,
                   yanchor="top", font=dict(size=13, color="#333"))
fig.add_annotation(x=3.1, y=0.5, text="<b>Accuracy</b><br>(TP + TN) / all<br>= 221 / 262 = 84%", showarrow=False,
                   xanchor="left", align="left", font=dict(size=13, color="#555"))
fig.update_xaxes(visible=False, range=[-1.25, 4.45])
fig.update_yaxes(visible=False, range=[-0.55, 3.3])
fig.update_layout(height=400, plot_bgcolor="rgba(0,0,0,0)", margin=dict(l=10, r=10, t=10, b=10))
fig

Next, we attach a cost to each kind of error. A missed churner costs the company a customer relationship, which for Prairie Wholesale means hundreds of dollars a year in lost margin. A false alarm costs one retention offer, perhaps a discount of tens of dollars. The two errors differ in cost by about a factor of ten. In the accuracy number, however, they count equally. A model tuned for accuracy is therefore tuned for an objective that does not match the business.

## Moving the threshold
As described in the logistic regression section, the logistic model first predicts a probability of churn for each customer, and `predict` converts this probability into a category with a threshold of 0.5. Most classification models work in the same way. In KNN, for example, the probability is the share of the k neighbors who churned, and the trees and forests of [Chapter 10](https://pyba.murtaza.cc/parts/part-04-predictive/ch-10-classification-2.html) also produce probabilities. The threshold of 0.5, however, is only a default, and it is not necessarily the best choice for the business. To use a different threshold, we first obtain the probabilities themselves with `predict_proba`:

In [ ]:
# --- Look: predict_proba for the first three test customers ---
logit.predict_proba(X_test)[:3].round(3)

`predict_proba` returns a NumPy array with one row per customer and two columns: the probability of class 0 (stay) and the probability of class 1 (churn). Each row sums to 1. The first test customer has a churn probability of 0.044. We keep the second column for every customer:

In [ ]:
proba = logit.predict_proba(X_test)[:, 1]     # P(churn) for each test customer

rows = []
for threshold in [0.5, 0.35, 0.2, 0.1]:
    flagged = proba >= threshold              # True for every customer we would call
    tp = (flagged & (y_test == 1)).sum()      # churners caught
    fp = (flagged & (y_test == 0)).sum()      # false alarms
    fn = ((~flagged) & (y_test == 1)).sum()   # churners missed
    cost = 60 * (tp + fp) + 600 * fn          # $60 per offer, $600 per lost customer
    rows.append([threshold, tp, fp, fn, f"${cost:,.0f}"])

pd.DataFrame(rows, columns=["threshold", "caught", "false alarms",
                            "missed", "error cost"])

- **`[:, 1]`.** A NumPy array is indexed by row and column. The colon means "all rows", so `[:, 1]` is column 1, the churn probabilities, for every customer. In the cell before it, `[:3]` is the first three rows.
- **`&` and `~`.** `flagged` holds one True or False per customer. With `&`, two such conditions are combined row by row, as in the [Chapter 3](https://pyba.murtaza.cc/parts/part-01-foundations/ch-03-pandas.html) filters. With `~`, a condition is negated, as in [Chapter 7](https://pyba.murtaza.cc/parts/part-03-statistical/ch-07-inference.html). Each condition is placed in its own parentheses.
- **`.sum()` on True/False values** returns the number of Trues, since True counts as 1.

In the cost formula, we assume that a retention offer costs \$60, that a lost customer costs \$600 in margin, and that a churner who receives an offer stays. At the default threshold, most of the error cost is due to the 33 missed churners (33 × \$600 = \$19,800). As the threshold falls, the model flags more customers. At a threshold of 0.1, the model flags 114 customers: 46 of the 58 churners and 68 customers who would have stayed. Because an offer is so much cheaper than a lost customer, the total error cost falls from \$21,780 to \$14,040.

We did not find the threshold of 0.1 by searching the table. We derived it from the two costs. If we flag a customer, we pay \$60 for certain. If we leave a customer alone, the expected loss is the churn probability times \$600. Flagging is the cheaper choice whenever $p \times 600 > 60$, that is, whenever $p > 60/600 = 0.1$. In general, the **break-even threshold** is the cost of an offer divided by the cost of a lost customer. The rule is valid only if the model's probabilities are accurate. With the threshold derived from the costs, we also avoid a subtle error. If we chose the threshold by searching for the lowest cost on the test set, the model would be tuned to those 262 customers. We could then no longer use the test set to measure performance on unseen data.

Setting the threshold is therefore a business calculation. The costs are set by the business, and the probabilities are estimated by the model.

The online edition has an interactive threshold explorer here, based on the same 262 test customers: the confusion matrix, recall, and total error cost change as the threshold moves, and the cost is lowest at thresholds far below 0.5, while accuracy is highest near 0.3. [Open the interactive explorer](https://pyba.murtaza.cc/assets/demos/threshold-confusion.html).


## Leakage
One of the hardest mistakes to notice in predictive work is **leakage**. Leakage occurs when a feature contains information from the period in which the label, the outcome that we intend to predict, is measured, that is, information that would not be available on the day the prediction is made. A model with a leaky feature scores higher on the test set than the same model without it, so the mistake appears as an improvement in the results. The snapshot was built to prevent leakage. To see its effect, we break the design on purpose. We recompute recency at the end of the data, in June 2026, so that the feature overlaps the label's 26-week window. Then we refit the model.

In [ ]:
orders = pd.read_csv(DATA_DIR / "pw_orders.csv", parse_dates=["order_date"])
data_end = orders["order_date"].max()      # the last order date in the data

# WRONG on purpose: recency measured at the end of the data, which
# includes the 26 weeks that define the churn label
last_order = orders.groupby("customer_id", as_index=False)["order_date"].max()
last_order["weeks_leaky"] = (data_end - last_order["order_date"]).dt.days / 7

leaky = snap[["customer_id"]].merge(last_order, on="customer_id", how="left")
X_leaky = X.copy()
X_leaky["weeks_leaky"] = leaky["weeks_leaky"]

# --- Look: the leaky feature for churners and for customers who stayed ---
check = pd.DataFrame({"churned": y, "weeks_leaky": X_leaky["weeks_leaky"]})
check.groupby("churned")["weeks_leaky"].agg(["min", "max"]).round(1)

In the code above, the leaky feature is built in four steps:

- **`data_end`** is the last order date in the data, at the end of the label's 26-week window.
- **`last_order`** holds each customer's most recent order date, found with `groupby` and `max`.
- **`weeks_leaky`** is the time from that order to `data_end`. The difference of two dates is a time difference, which is converted to a number of days with `.dt.days` and then divided by 7 to give weeks.
- **`leaky`** is the result of lining these values up with the snapshot customers, using `merge` as in [Chapter 3](https://pyba.murtaza.cc/parts/part-01-foundations/ch-03-pandas.html). It therefore has the same 872 rows, in the same order, as `X`, so its column can be added to a copy of `X` directly.

The table shows the problem before any model is fitted. Every customer who stayed has a leaky recency of at most 24.4 weeks, and every churner has one of at least 26.0 weeks. The feature is therefore a restatement of the outcome label. We fit the same pipeline with it:

In [ ]:
Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky, y, test_size=0.3, stratify=y, random_state=0)

leaky_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
leaky_model.fit(Xl_train, yl_train)
print(f"test accuracy with the leaky feature: "
      f"{accuracy_score(yl_test, leaky_model.predict(Xl_test)):.3f}")

The leaky model reaches 100% test accuracy. It has learned the definition of the label: customers with no orders in the last 26 weeks are, by construction, the churners. In practice, this model would be useless. At the time of prediction, the future orders on which the leaky feature depends have not been placed, so the feature cannot be computed as it was in training, and the pattern learned from it no longer holds. Leakage is prevented by the rule under which the snapshot was built: **every feature must be computable on the day the prediction is made.** If a model's test numbers look implausibly good, we check for leakage first.

## Evaluation against two baselines

|  |  |
|---|---|
| **Metric** | total error cost on the test set, with \$60 per offer and \$600 per lost customer, assuming an offer keeps a churner. |
| **Test set** | the 262 held-out customers. |
| **Baselines** | no model and no offers, so every churner is lost; and an offer to every customer, so no churner is lost. |



In [ ]:
def error_cost(flagged):
    tp = (flagged & (y_test == 1)).sum()
    fp = (flagged & (y_test == 0)).sum()
    fn = ((~flagged) & (y_test == 1)).sum()
    return 60 * (tp + fp) + 600 * fn

pd.Series({
    "no offers (every churner lost)": f"${600 * (y_test == 1).sum():,}",
    "offer to every customer": f"${60 * len(y_test):,}",
    "model at threshold 0.5": f"${error_cost(proba >= 0.5):,}",
    "model at threshold 0.1": f"${error_cost(proba >= 0.1):,}",
}).to_frame("test-set error cost")

With no offers, the company loses all 58 churners, at a cost of \$34,800. An offer to every customer costs \$15,720. We need both baselines, because a model has to improve on the better of the two policies that require no model. At the default threshold of 0.5, the model costs \$21,780. This is less than sending no offers, but it is more than the offer-to-everyone policy, which requires no model at all. At the break-even threshold of 0.1, the model costs \$14,040, the lowest of the four. It is about 60% below the no-offer cost and about 11% below the offer-to-everyone cost, measured on customers on which the model was never trained. The model's advantage over a blanket offer, \$1,680 on 262 customers, is therefore modest. The saving is due to the 148 customers who receive no offer, of whom 136 stay.

The cost figures assume that an offer keeps a churner. This assumption cannot be established from the model. It can be tested with the experiment design from [Chapter 7](https://pyba.murtaza.cc/parts/part-03-statistical/ch-07-inference.html): send retention offers to a random half of the flagged customers, and compare churn in the two halves. We then use the two stages of the course together: prediction to find the at-risk customers, and inference to test whether churn is lower among the customers who receive the offer.

## The decision this informs

The account team receives a weekly list of every customer whose churn probability is above the break-even threshold, sorted by twelve-month revenue so that the most valuable relationships are called first. The threshold is derived from the costs and must be recomputed whenever they change. For example, if the offer cost rose from \$60 to \$120, the break-even threshold would rise from 0.1 to 0.2, and the list would be shorter. The team is also told the model's limits. Because the model scores customers on ordering behavior alone, it cannot anticipate a customer who leaves for a reason absent from the orders, such as a competitor opening nearby.

## Exercises


### Build lab

Build a churn model using only three features: `weeks_since_last_order`, `orders_12m`, and `tenure_months`. Fit the logistic pipeline on the same split (`random_state=0`), and report the test accuracy, the confusion matrix, and the recall. Compare it with the full model in two sentences: how much did the other features add?

### Evaluate lab

Using your three-feature model's probabilities, build the cost table for thresholds 0.5, 0.35, 0.2, and 0.1 with the chapter's \$60 and \$600 figures. Then recompute it with a \$300 loss figure. State in two sentences how the break-even threshold changes under the cheaper loss figure, and why the threshold is a business decision, with the model supplying only the probabilities.

A starter notebook for this lab is provided. It restates the task, reproduces the objects from this chapter that the lab builds on, and marks the cells you complete. [**Open ch-09-lab-starter.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-09-lab-starter.ipynb), or download it from the course page in Blackboard. Colab opens a notebook from GitHub read-only: click **Copy to Drive** in the toolbar before you edit anything, and work in that copy.
